# SQL Queries para Dashboards

QueriesSQL prontas para criar dashboards executivos.

## Dashboards Recomendados:
1. **CRM Dashboard**: Visão geral de clientes e churn
2. **Campaign Performance**: Performance de campanhas
3. **Growth Dashboard**: Métricas de crescimento
4. **Executive Summary**: KPIs principais

Copie as queries abaixo para criar dashboards no Databricks SQL.

In [0]:
%sql
-- Dashboard 1: Churn Risk Overview
-- Top clientes com maior risco de churn

SELECT 
  cs.customer_id,
  c.segment,
  c.customer_age_days,
  cf.recency_days,
  cf.frequency,
  cf.monetary_total,
  cs.churn_probability,
  cs.churn_risk_category,
  cs.customer_value_score,
  cs.engagement_score
FROM workspace.customer_intelligence_gold.customer_scores cs
INNER JOIN workspace.customer_intelligence_silver.customers c ON cs.customer_id = c.customer_id
INNER JOIN workspace.customer_intelligence_gold.customer_features cf ON cs.customer_id = cf.customer_id
WHERE cs.churn_risk_category = 'High'
  AND c.is_active = 1
ORDER BY cs.churn_probability DESC, cs.customer_value_score DESC
LIMIT 100;

In [0]:
%sql
-- Dashboard 2: Distribuição de Risco por Segmento

SELECT 
  c.segment,
  cs.churn_risk_category,
  COUNT(*) as customer_count,
  AVG(cs.churn_probability) as avg_churn_prob,
  AVG(cs.customer_value_score) as avg_value_score,
  SUM(cf.monetary_total) as total_revenue_at_risk
FROM workspace.customer_intelligence_gold.customer_scores cs
INNER JOIN workspace.customer_intelligence_silver.customers c ON cs.customer_id = c.customer_id
INNER JOIN workspace.customer_intelligence_gold.customer_features cf ON cs.customer_id = cf.customer_id
GROUP BY c.segment, cs.churn_risk_category
ORDER BY c.segment, cs.churn_risk_category;

In [0]:
%sql
-- Dashboard 3: Campaign Performance

SELECT 
  campaign_id,
  campaign_name,
  Control_conversion_rate,
  Treatment_conversion_rate,
  lift_pct,
  uplift_absolute,
  incremental_revenue,
  is_significant_chi2 as is_significant,
  result_category,
  CASE 
    WHEN result_category = 'Strong Positive' THEN '🟢'
    WHEN result_category = 'Positive' THEN '🟡'
    WHEN result_category = 'Negative' THEN '🔴'
    ELSE '⚪'
  END as status_icon
FROM workspace.customer_intelligence_gold.campaign_ab_test_results
ORDER BY lift_pct DESC;

In [0]:
%sql
-- Dashboard 4: ROAS (Return on Ad Spend)

SELECT 
  campaign_id,
  campaign_name,
  budget,
  incremental_revenue,
  roas,
  roi_pct,
  result_category,
  CASE 
    WHEN roas >= 3 THEN 'Excellent (3x+)'
    WHEN roas >= 2 THEN 'Good (2-3x)'
    WHEN roas >= 1 THEN 'Breakeven (1-2x)'
    ELSE 'Loss (<1x)'
  END as roas_category
FROM workspace.customer_intelligence_gold.campaign_roas
WHERE result_category IN ('Positive', 'Strong Positive')
ORDER BY roas DESC;

In [0]:
%sql
-- Dashboard 5: Executive KPIs

SELECT 
  'Total Customers' as metric,
  total_customers as value,
  '' as change_pct
FROM workspace.customer_intelligence_gold.business_kpis_history
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM workspace.customer_intelligence_gold.business_kpis_history)

UNION ALL

SELECT 
  'Active Customers',
  active_customers,
  CAST(ROUND((active_customers * 100.0 / total_customers), 1) AS STRING) || '%'
FROM workspace.customer_intelligence_gold.business_kpis_history
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM workspace.customer_intelligence_gold.business_kpis_history)

UNION ALL

SELECT 
  'Churn Rate',
  churned_customers,
  CAST(churn_rate_pct AS STRING) || '%'
FROM workspace.customer_intelligence_gold.business_kpis_history
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM workspace.customer_intelligence_gold.business_kpis_history)

UNION ALL

SELECT 
  'Total Revenue',
  CAST(total_revenue AS BIGINT),
  ''
FROM workspace.customer_intelligence_gold.business_kpis_history
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM workspace.customer_intelligence_gold.business_kpis_history);

In [0]:
%sql
-- Dashboard 6: RFM Segmentation

SELECT 
  c.segment,
  COUNT(*) as customer_count,
  AVG(rf.recency_days) as avg_recency,
  AVG(rf.frequency) as avg_frequency,
  AVG(rf.monetary_total) as avg_monetary,
  SUM(rf.monetary_total) as total_revenue,
  AVG(cs.churn_probability) as avg_churn_risk,
  AVG(cs.engagement_score) as avg_engagement
FROM workspace.customer_intelligence_silver.customers c
INNER JOIN workspace.customer_intelligence_gold.rfm_features rf ON c.customer_id = rf.customer_id
INNER JOIN workspace.customer_intelligence_gold.customer_scores cs ON c.customer_id = cs.customer_id
WHERE c.is_active = 1
GROUP BY c.segment
ORDER BY total_revenue DESC;

In [0]:
%sql
-- Dashboard 7: Monthly Trends

SELECT 
  year_month,
  total_exposures,
  total_conversions,
  ROUND(conversion_rate, 2) as conversion_rate_pct,
  LAG(conversion_rate) OVER (ORDER BY year_month) as prev_month_rate,
  ROUND(conversion_rate - LAG(conversion_rate) OVER (ORDER BY year_month), 2) as rate_change
FROM workspace.customer_intelligence_gold.campaign_performance_trend
ORDER BY year_month DESC
LIMIT 12;